# ⚡ CoinGecko Crypto Market Data Lake — Lambda Architecture
**Source:** CoinGecko API (free tier, no API key — https://api.coingecko.com)  
**Architecture:** Lambda (Speed Layer + Batch Layer → Serving Layer)  
**Stack:** CoinGecko API → Kafka → PySpark Streaming (speed) + PySpark Batch (batch)  
           → Cassandra (serving layer) → Snowflake (analytical layer)  
**Orchestration:** Apache Airflow  

## Overview
Advanced Lambda Architecture pipeline for cryptocurrency market data:
- **Speed layer:** Real-time price streaming every 60 seconds → Cassandra for low-latency reads
- **Batch layer:** Historical OHLCV data daily → Snowflake for analytical queries
- **Serving layer:** Cassandra merges real-time + batch views for unified query interface

**Top 50 coins tracked:** BTC, ETH, BNB, SOL, ADA, XRP, DOGE, DOT, MATIC, and 41 more  
**Metrics:** price, market cap, 24h volume, 24h change%, ATH, circulating supply, OHLCV

## Section 1 — Imports & Configuration

In [ ]:
# pip install requests kafka-python pyspark cassandra-driver snowflake-connector-python pandas numpy

import requests
import json
import time
import logging
import hashlib
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta, date
from kafka import KafkaProducer
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra import ConsistencyLevel
from cassandra.query import BatchStatement
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, to_timestamp, window, avg, max as spark_max,
    min as spark_min, first, last, count, lit,
    round as spark_round, current_timestamp, udf,
    percent_rank, lag, when
)
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType, BooleanType
)
from pyspark.sql.window import Window
import snowflake.connector

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('LambdaCryptoPipeline')
print('✅ All imports loaded')

In [ ]:
# ── CoinGecko API ───────────────────────────────────────────────
# Free tier: 10-30 calls/min, no API key needed
CG_BASE_URL    = 'https://api.coingecko.com/api/v3'
CG_RATE_LIMIT  = 1.5   # seconds between requests
TOP_N_COINS    = 50

# ── Kafka ───────────────────────────────────────────────────────
KAFKA_BOOTSTRAP          = 'localhost:9092'
KAFKA_TOPIC_PRICES       = 'crypto-prices-realtime'
KAFKA_TOPIC_OHLCV        = 'crypto-ohlcv-batch'

# ── Cassandra (Serving Layer — low-latency reads) ───────────────
CASSANDRA_HOSTS     = ['localhost']
CASSANDRA_PORT      = 9042
CASSANDRA_KEYSPACE  = 'crypto_serving'

# ── Snowflake (Analytical Layer) ────────────────────────────────
SNOWFLAKE_CONFIG = {
    'account':   'your_account.ap-southeast-1',
    'user':      'your_username',
    'password':  'your_password',
    'warehouse': 'COMPUTE_WH',
    'database':  'CRYPTO_DW',
    'schema':    'ANALYTICS',
    'role':      'SYSADMIN',
}

print(f'✅ Lambda Architecture config ready')
print(f'   Speed layer  : Kafka → PySpark → Cassandra')
print(f'   Batch layer  : CoinGecko OHLCV → Kafka → PySpark → Snowflake')
print(f'   Serving layer: Cassandra (merged real-time + batch views)')

## Section 2 — Speed Layer: Real-Time Price Producer

In [ ]:
def get_top_coin_ids(n: int = 50) -> list:
    """Fetch top-N coin IDs by market cap from CoinGecko."""
    resp = requests.get(
        f'{CG_BASE_URL}/coins/markets',
        params={
            'vs_currency': 'usd', 'order': 'market_cap_desc',
            'per_page': n, 'page': 1, 'sparkline': False
        },
        timeout=15
    )
    resp.raise_for_status()
    return [c['id'] for c in resp.json()]

def fetch_realtime_prices(coin_ids: list) -> list:
    """
    Fetch current price + market data for all coins.
    CoinGecko /simple/price endpoint: free, no key needed.
    """
    chunk_size = 25  # API limit per request
    all_records = []
    now = datetime.now(timezone.utc)

    for i in range(0, len(coin_ids), chunk_size):
        chunk = coin_ids[i:i + chunk_size]
        try:
            resp = requests.get(
                f'{CG_BASE_URL}/coins/markets',
                params={
                    'ids':         ','.join(chunk),
                    'vs_currency': 'usd',
                    'order':       'market_cap_desc',
                    'per_page':    50,
                    'page':        1,
                    'sparkline':   False,
                    'price_change_percentage': '1h,24h,7d'
                },
                timeout=15
            )
            resp.raise_for_status()
            for coin in resp.json():
                all_records.append({
                    'coin_id':              coin['id'],
                    'symbol':               coin['symbol'].upper(),
                    'name':                 coin['name'],
                    'current_price_usd':    coin.get('current_price'),
                    'market_cap_usd':       coin.get('market_cap'),
                    'market_cap_rank':      coin.get('market_cap_rank'),
                    'volume_24h_usd':       coin.get('total_volume'),
                    'price_change_1h_pct':  coin.get('price_change_percentage_1h_in_currency'),
                    'price_change_24h_pct': coin.get('price_change_percentage_24h'),
                    'price_change_7d_pct':  coin.get('price_change_percentage_7d_in_currency'),
                    'ath_usd':              coin.get('ath'),
                    'ath_change_pct':       coin.get('ath_change_percentage'),
                    'circulating_supply':   coin.get('circulating_supply'),
                    'total_supply':         coin.get('total_supply'),
                    'last_updated':         coin.get('last_updated', now.isoformat()),
                    'ingested_at':          now.isoformat(),
                    'layer':                'speed',
                    'source':               'coingecko-api'
                })
            time.sleep(CG_RATE_LIMIT)
        except Exception as e:
            logger.warning(f'Chunk {i//chunk_size + 1} error: {e}')

    return all_records

def stream_prices_to_kafka(iterations: int = 5, interval_sec: int = 60):
    """
    SPEED LAYER: Continuously poll CoinGecko prices and publish to Kafka.
    Default: 5 iterations × 60s = 5 minutes of streaming.
    """
    coin_ids = get_top_coin_ids(TOP_N_COINS)
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        acks='all', compression_type='gzip'
    )
    total_published = 0
    for i in range(iterations):
        prices = fetch_realtime_prices(coin_ids)
        for p in prices:
            producer.send(
                KAFKA_TOPIC_PRICES,
                key=p['coin_id'].encode(),
                value=p
            )
            total_published += 1
        producer.flush()
        logger.info(f'⚡ Iteration {i+1}/{iterations}: {len(prices)} prices published')
        if i < iterations - 1:
            time.sleep(interval_sec)

    producer.close()
    logger.info(f'✅ Speed layer: {total_published} total price events published')
    return total_published

# stream_prices_to_kafka(iterations=3, interval_sec=60)

## Section 3 — Batch Layer: Historical OHLCV Producer

In [ ]:
def fetch_ohlcv_history(coin_id: str, days: int = 90) -> list:
    """
    BATCH LAYER: Fetch OHLCV history for a single coin.
    CoinGecko /coins/{id}/ohlc endpoint — free, no key.
    Returns daily OHLCV candles.
    """
    resp = requests.get(
        f'{CG_BASE_URL}/coins/{coin_id}/ohlc',
        params={'vs_currency': 'usd', 'days': days},
        timeout=20
    )
    resp.raise_for_status()
    candles = resp.json()
    now = datetime.now(timezone.utc).isoformat()
    records = []
    for candle in candles:
        # candle = [timestamp_ms, open, high, low, close]
        ts = datetime.fromtimestamp(candle[0] / 1000, tz=timezone.utc)
        records.append({
            'coin_id':      coin_id,
            'candle_date':  ts.date().isoformat(),
            'timestamp_ms': candle[0],
            'open':         candle[1],
            'high':         candle[2],
            'low':          candle[3],
            'close':        candle[4],
            'price_range':  round(candle[2] - candle[3], 6),
            'is_bullish':   candle[4] >= candle[1],
            'ingested_at':  now,
            'batch_date':   str(date.today()),
            'layer':        'batch',
            'source':       'coingecko-ohlcv'
        })
    return records

def batch_ohlcv_to_kafka(coin_ids: list = None, days: int = 90) -> int:
    """
    BATCH LAYER: Fetch OHLCV for top 20 coins (rate-limit friendly)
    and publish to Kafka batch topic.
    """
    if coin_ids is None:
        coin_ids = get_top_coin_ids(20)  # top 20 for batch

    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        acks='all', compression_type='gzip'
    )
    total = 0
    for coin_id in coin_ids:
        try:
            candles = fetch_ohlcv_history(coin_id, days=days)
            for c in candles:
                producer.send(
                    KAFKA_TOPIC_OHLCV,
                    key=f"{coin_id}_{c['candle_date']}".encode(),
                    value=c
                )
                total += 1
            logger.info(f'  ✅ {coin_id}: {len(candles)} OHLCV candles published')
            time.sleep(CG_RATE_LIMIT)
        except Exception as e:
            logger.warning(f'  ⚠️  {coin_id}: {e}')

    producer.flush()
    producer.close()
    logger.info(f'✅ Batch layer: {total} OHLCV candles published')
    return total

# batch_ohlcv_to_kafka(days=90)

## Section 4 — Speed Processing: PySpark → Cassandra

In [ ]:
spark = SparkSession.builder \
    .appName('CryptoLambdaSpeedLayer') \
    .config('spark.jars.packages',
            'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,'
            'com.datastax.spark:spark-cassandra-connector_2.12:3.4.1') \
    .config('spark.cassandra.connection.host', 'localhost') \
    .config('spark.cassandra.connection.port', '9042') \
    .config('spark.sql.streaming.checkpointLocation', '/tmp/checkpoints/crypto_speed') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')

PRICE_SCHEMA = StructType([
    StructField('coin_id',              StringType(),  True),
    StructField('symbol',               StringType(),  True),
    StructField('name',                 StringType(),  True),
    StructField('current_price_usd',    DoubleType(),  True),
    StructField('market_cap_usd',       DoubleType(),  True),
    StructField('market_cap_rank',      IntegerType(), True),
    StructField('volume_24h_usd',       DoubleType(),  True),
    StructField('price_change_1h_pct',  DoubleType(),  True),
    StructField('price_change_24h_pct', DoubleType(),  True),
    StructField('price_change_7d_pct',  DoubleType(),  True),
    StructField('ath_usd',              DoubleType(),  True),
    StructField('ath_change_pct',       DoubleType(),  True),
    StructField('circulating_supply',   DoubleType(),  True),
    StructField('last_updated',         StringType(),  True),
    StructField('ingested_at',          StringType(),  True),
    StructField('layer',                StringType(),  True),
])

def classify_volatility(pct_24h: float) -> str:
    if pct_24h is None: return 'unknown'
    abs_p = abs(pct_24h)
    if abs_p >= 10: return 'extreme'
    if abs_p >= 5:  return 'high'
    if abs_p >= 2:  return 'medium'
    return 'low'

volatility_udf = udf(classify_volatility, StringType())

# Read from Kafka speed topic
price_stream = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP) \
    .option('subscribe', KAFKA_TOPIC_PRICES) \
    .option('startingOffsets', 'latest') \
    .option('maxOffsetsPerTrigger', 200) \
    .load()

enriched_stream = price_stream \
    .select(from_json(col('value').cast('string'), PRICE_SCHEMA).alias('d')) \
    .select('d.*') \
    .withColumn('event_time',        to_timestamp(col('last_updated'))) \
    .withColumn('volatility_class',  volatility_udf(col('price_change_24h_pct'))) \
    .withColumn('is_top10',
        when(col('market_cap_rank') <= 10, True).otherwise(False)
    ) \
    .withColumn('processed_at', current_timestamp())

def write_to_cassandra_speed(batch_df, batch_id: int):
    """
    Write real-time price events to Cassandra serving layer.
    Cassandra schema optimized for time-series reads by coin_id.
    """
    rows = batch_df.toJSON().collect()
    if not rows:
        return
    cluster = Cluster(CASSANDRA_HOSTS, port=CASSANDRA_PORT)
    session = cluster.connect(CASSANDRA_KEYSPACE)
    insert_cql = """
        INSERT INTO price_time_series (
            coin_id, event_time, symbol, name,
            current_price_usd, market_cap_usd, volume_24h_usd,
            price_change_24h_pct, volatility_class, is_top10, ingested_at
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?)
        USING TTL 2592000
    """  # TTL 30 days
    stmt = session.prepare(insert_cql)
    batch = BatchStatement(consistency_level=ConsistencyLevel.LOCAL_QUORUM)
    count = 0
    for row_json in rows:
        r = json.loads(row_json)
        batch.add(stmt, [
            r.get('coin_id'), r.get('event_time'), r.get('symbol'), r.get('name'),
            r.get('current_price_usd'), r.get('market_cap_usd'), r.get('volume_24h_usd'),
            r.get('price_change_24h_pct'), r.get('volatility_class'),
            r.get('is_top10'), r.get('ingested_at')
        ])
        count += 1
        if count % 50 == 0:  # execute batch every 50 rows
            session.execute(batch)
            batch = BatchStatement(consistency_level=ConsistencyLevel.LOCAL_QUORUM)
    if count % 50 != 0:
        session.execute(batch)
    cluster.shutdown()
    logger.info(f'⚡ Batch {batch_id}: {count} rows → Cassandra')

speed_query = enriched_stream.writeStream \
    .foreachBatch(write_to_cassandra_speed) \
    .outputMode('append') \
    .trigger(processingTime='60 seconds') \
    .start()

print('⚡ Speed layer running — writing to Cassandra every 60s')
# speed_query.awaitTermination()

## Section 5 — Cassandra Schema & Keyspace Setup

In [ ]:
CASSANDRA_DDL = """
-- Keyspace
CREATE KEYSPACE IF NOT EXISTS crypto_serving
WITH REPLICATION = {
    'class': 'SimpleStrategy',
    'replication_factor': 1
};

-- Speed layer table: time-series per coin
-- Partition by coin_id, cluster by event_time DESC for latest-first reads
CREATE TABLE IF NOT EXISTS crypto_serving.price_time_series (
    coin_id              TEXT,
    event_time           TIMESTAMP,
    symbol               TEXT,
    name                 TEXT,
    current_price_usd    DOUBLE,
    market_cap_usd       DOUBLE,
    volume_24h_usd       DOUBLE,
    price_change_24h_pct DOUBLE,
    volatility_class     TEXT,
    is_top10             BOOLEAN,
    ingested_at          TEXT,
    PRIMARY KEY (coin_id, event_time)
) WITH CLUSTERING ORDER BY (event_time DESC)
  AND default_time_to_live = 2592000;  -- 30-day TTL

-- Batch layer table: daily aggregates merged from batch processing
CREATE TABLE IF NOT EXISTS crypto_serving.daily_ohlcv (
    coin_id       TEXT,
    candle_date   DATE,
    open_price    DOUBLE,
    high_price    DOUBLE,
    low_price     DOUBLE,
    close_price   DOUBLE,
    price_range   DOUBLE,
    is_bullish    BOOLEAN,
    batch_date    TEXT,
    loaded_at     TEXT,
    PRIMARY KEY (coin_id, candle_date)
) WITH CLUSTERING ORDER BY (candle_date DESC);

-- Serving view: latest price per coin (for dashboard)
CREATE TABLE IF NOT EXISTS crypto_serving.latest_price_view (
    coin_id              TEXT PRIMARY KEY,
    symbol               TEXT,
    name                 TEXT,
    current_price_usd    DOUBLE,
    market_cap_usd       DOUBLE,
    market_cap_rank      INT,
    price_change_24h_pct DOUBLE,
    volatility_class     TEXT,
    last_updated         TEXT
);
"""

def setup_cassandra_schema():
    """Create keyspace and tables if they don't exist."""
    cluster = Cluster(CASSANDRA_HOSTS, port=CASSANDRA_PORT)
    session = cluster.connect()
    for stmt in [s.strip() for s in CASSANDRA_DDL.split(';') if s.strip() and not s.strip().startswith('--')]:
        try:
            session.execute(stmt)
        except Exception as e:
            logger.warning(f'DDL: {e}')
    cluster.shutdown()
    logger.info('✅ Cassandra schema ready')

# setup_cassandra_schema()

## Section 6 — Batch Processing: OHLCV → Snowflake

In [ ]:
def process_ohlcv_batch_to_snowflake(execution_date: str) -> int:
    """
    BATCH LAYER: Read OHLCV from Kafka, compute technical indicators,
    write enriched data to Snowflake for analytical queries.
    """
    OHLCV_SCHEMA = StructType([
        StructField('coin_id',      StringType(),  True),
        StructField('candle_date',  StringType(),  True),
        StructField('timestamp_ms', DoubleType(),  True),
        StructField('open',         DoubleType(),  True),
        StructField('high',         DoubleType(),  True),
        StructField('low',          DoubleType(),  True),
        StructField('close',        DoubleType(),  True),
        StructField('price_range',  DoubleType(),  True),
        StructField('is_bullish',   BooleanType(), True),
        StructField('batch_date',   StringType(),  True),
        StructField('ingested_at',  StringType(),  True),
    ])

    raw = spark.read.format('kafka') \
        .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP) \
        .option('subscribe', KAFKA_TOPIC_OHLCV) \
        .option('startingOffsets', 'earliest') \
        .option('endingOffsets', 'latest') \
        .load()

    df = raw.select(from_json(col('value').cast('string'), OHLCV_SCHEMA).alias('d')).select('d.*') \
        .dropDuplicates(['coin_id', 'candle_date'])

    # Technical indicators via window functions
    w = Window.partitionBy('coin_id').orderBy('candle_date')
    df_enriched = df \
        .withColumn('prev_close',   lag('close', 1).over(w)) \
        .withColumn('daily_return_pct', spark_round(
            (col('close') - col('prev_close')) / col('prev_close') * 100, 4
        )) \
        .withColumn('ma_7d', spark_round(
            avg('close').over(w.rowsBetween(-6, 0)), 6
        )) \
        .withColumn('ma_30d', spark_round(
            avg('close').over(w.rowsBetween(-29, 0)), 6
        )) \
        .withColumn('high_low_ratio', spark_round(col('high') / col('low'), 4)) \
        .withColumn('golden_cross',
            when((col('ma_7d') > col('ma_30d')) &
                 (lag('ma_7d', 1).over(w) <= lag('ma_30d', 1).over(w)), True
            ).otherwise(False)
        ) \
        .withColumn('loaded_at', current_timestamp()) \
        .drop('prev_close')

    # Write to Snowflake
    count = df_enriched.count()
    pandas_df = df_enriched.toPandas()

    conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
    from snowflake.connector.pandas_tools import write_pandas
    pandas_df.columns = [c.upper() for c in pandas_df.columns]
    pandas_df['BATCH_DATE'] = execution_date
    write_pandas(conn, pandas_df, 'CRYPTO_OHLCV_HISTORY',
                 database='CRYPTO_DW', schema='ANALYTICS')
    conn.close()

    logger.info(f'✅ OHLCV batch: {count} candles → Snowflake')
    return count

# process_ohlcv_batch_to_snowflake(str(date.today()))

## Section 7 — Cassandra Serving Queries

In [ ]:
def get_latest_prices(top_n: int = 20) -> pd.DataFrame:
    """Query Cassandra serving layer for latest prices — millisecond latency."""
    cluster = Cluster(CASSANDRA_HOSTS, port=CASSANDRA_PORT)
    session = cluster.connect(CASSANDRA_KEYSPACE)
    rows = session.execute(
        'SELECT coin_id, symbol, current_price_usd, market_cap_usd, '
        'price_change_24h_pct, volatility_class, last_updated '
        'FROM latest_price_view LIMIT %s', (top_n,)
    )
    cluster.shutdown()
    df = pd.DataFrame(list(rows))
    df.sort_values('market_cap_usd', ascending=False, inplace=True)
    print(f'\n📊 Latest prices (top {top_n}):')
    print(df[['symbol','current_price_usd','price_change_24h_pct','volatility_class']].to_string(index=False))
    return df

def get_price_history(coin_id: str, hours: int = 24) -> pd.DataFrame:
    """Get time-series for one coin from Cassandra speed layer."""
    cluster = Cluster(CASSANDRA_HOSTS, port=CASSANDRA_PORT)
    session = cluster.connect(CASSANDRA_KEYSPACE)
    cutoff  = datetime.now(timezone.utc) - timedelta(hours=hours)
    rows    = session.execute(
        'SELECT event_time, current_price_usd, volume_24h_usd, '
        'price_change_24h_pct FROM price_time_series '
        'WHERE coin_id = %s AND event_time >= %s', (coin_id, cutoff)
    )
    cluster.shutdown()
    df = pd.DataFrame(list(rows))
    print(f'\n📈 {coin_id.upper()} last {hours}h: {len(df)} data points')
    return df

# get_latest_prices(20)
# get_price_history('bitcoin', hours=24)

## Section 8 — Lambda Architecture Summary

| Layer | Technology | Latency | Use Case |
|---|---|---|---|
| **Speed (real-time)** | Kafka → PySpark Streaming → Cassandra | < 60s | Live prices, alerts |
| **Batch (historical)** | CoinGecko OHLCV → Kafka → PySpark → Snowflake | Daily | Analytics, backtesting |
| **Serving (merged)** | Cassandra `latest_price_view` | ms | Dashboard, API queries |
| **Analytical** | Snowflake + technical indicators | Minutes | Reports, trend analysis |

**Volume:**  
- Speed layer: 50 coins × every 60s = ~4,320,000 events/day  
- Batch layer: 20 coins × 90 candles = 1,800 OHLCV records/run  

**Cassandra design:**  
- Partitioned by `coin_id` → all time-series for a coin on same node  
- Clustered by `event_time DESC` → latest-first reads O(1)  
- 30-day TTL auto-expires old data → no manual cleanup  

**Why Lambda?**  
Serves two competing needs: millisecond price lookups (speed) and complex analytical queries
with technical indicators (batch), unified through Cassandra's serving layer.